# 2.3 Preprocesamiento y entrenamiento de modelos
En este notebook nos centraremos en el preprocesamiento de los datos para preparar el dataset para modelos de clasificación.
Después aplicaremos algoritmos de clasificación

Generamos dos datasets transformados:

1. **Para árboles / Random Forest** → `OrdinalEncoder` en categóricas, **sin escalado** en numéricas.
2. **Para SVM / KNN / Regresión Logística** → `OneHotEncoder` en categóricas y **StandardScaler** en numéricas.

Se separa train/test estratificado y se guardan los resultados

In [17]:
import pandas as pd
import numpy as np

# Cargar dataset
df = pd.read_csv('adult-1.csv', sep=";", header=None)
df.columns = columns = ["age", "workclass", "fnlwgt", "education", "education_num",
                        "marital_status", "occupation", "relationship", "race", "sex",
                        "capital_gain", "capital_loss", "hours_per_week", "native_country", "income"]

## Limpieza de datos

In [18]:
# Reemplazar valores '?' por NaN
df = df.replace(['?', ' ?'], np.nan)
# Imputar faltantes: numéricas con la media, categóricas con la moda

for col in df.columns:
    if df[col].dtype == 'object':
        df[col] = df[col].fillna(df[col].mode()[0])
    else:
        df[col] = df[col].fillna(df[col].mean())

print(f"Tamaño del dataset después de imputación: {df.shape}")
print(f"Valores faltantes restantes: {df.isna().sum().sum()}")

Tamaño del dataset después de imputación: (32561, 15)
Valores faltantes restantes: 0


## Separación de variables

In [19]:
# Separamos las variables predictoras (X) de las objetivos (y)
X = df.drop('income', axis=1)
y = df['income']
# Separamos las variables numéricas de las categóricas
cat_cols = X.select_dtypes(include=['object']).columns.tolist()
num_cols = X.select_dtypes(exclude=['object']).columns.tolist()

print(f"Variables categóricas: {cat_cols}")
print(f"Variables numéricas: {num_cols}")

Variables categóricas: ['workclass', 'education', 'marital_status', 'occupation', 'relationship', 'race', 'sex', 'native_country']
Variables numéricas: ['age', 'fnlwgt', 'education_num', 'capital_gain', 'capital_loss', 'hours_per_week']


## Split Train / Test (estratificado)

In [20]:
from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test = train_test_split(
    X, y,
    test_size=0.2,
    random_state=42,
    stratify=y
)

print(X_train.shape, X_test.shape, y_train.shape, y_test.shape)

(26048, 14) (6513, 14) (26048,) (6513,)


## Dataset 1: Árboles / Random Forest (OrdinalEncoder, sin escalado)

## Tratamiento recomendado para Random Forest y Árboles de Decisión

### Variables numéricas
  - Se dejan como están.  
  - Solo imputar valores faltantes (media, mediana o moda).  

### Variables categóricas
  - Codificarlas con OrdinalEncoder o LabelEncoder (cada categoría se transforma en un número entero).  
  - No usar OneHotEncoder salvo que haya muy pocas categorías → porque aumenta mucho la dimensionalidad y no aporta mejora a los árboles.  

### Valores faltantes
  - Imputar (con la moda o crear categoría "Missing").  
  - Nota: algunos algoritmos como XGBoost y LightGBM manejan NaN de forma nativa.  


In [21]:
# ============================================================
# DATASET 1 → Random Forest / Árboles de Decisión
# ============================================================

from sklearn.preprocessing import OrdinalEncoder

X_train_tree = X_train.copy()
X_test_tree = X_test.copy()

encoder = OrdinalEncoder(handle_unknown='use_encoded_value', unknown_value=-1)

X_train_tree[cat_cols] = encoder.fit_transform(X_train_tree[cat_cols])
X_test_tree[cat_cols] = encoder.transform(X_test_tree[cat_cols])

print(f"Dataset Árboles: {X_train_tree.shape} {X_test_tree.shape}")

Dataset Árboles: (26048, 14) (6513, 14)


## Dataset 2: SVM / KNN / Regresión Logística

## Tratamiento recomendado para Modelos Lineales, SVM y KNN

### Variables numéricas
- Requieren **escalado** (por ejemplo con `StandardScaler` o `MinMaxScaler`).  
- Imputar valores faltantes (media o mediana).  

### Variables categóricas
- Usar **OneHotEncoder** para evitar asignar un orden artificial a las categorías.  
- Adecuado porque estos modelos dependen de distancias y pesos lineales.  

### Valores faltantes
- Imputar con la **moda** en categóricas.  
- En algunos casos se puede añadir una categoría `"Missing"`.  

### Notas
- El escalado debe aplicarse **después** de la codificación OneHot.  
- Estos modelos son más sensibles a alta dimensionalidad, por lo que conviene eliminar variables irrelevantes o hacer selección de características.

In [22]:
# ============================================================
# DATASET 2 → Modelos lineales / SVM / KNN
# ============================================================

# OneHot para categóricas
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.compose import ColumnTransformer
# Escalar solo numéricas
preprocessor = ColumnTransformer(
    transformers=[
        ('num', StandardScaler(), num_cols),
        ('cat', OneHotEncoder(handle_unknown='ignore', sparse_output=False), cat_cols)
    ])
# Concatenar numéricas + categóricas
X_train_lin = preprocessor.fit_transform(X_train)
X_test_lin = preprocessor.transform(X_test)

print(f"Dataset Lineales: {X_train_lin.shape} {X_test_lin.shape}")

Dataset Lineales: (26048, 105) (6513, 105)


## Modelos de clasificación

## Dataset 1

In [23]:
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score, confusion_matrix, classification_report

models_tree = {
    'Árbol de Decisión': DecisionTreeClassifier(random_state=42),
    'Random Forest': RandomForestClassifier(random_state=42)
}

for name, model in models_tree.items():
    model.fit(X_train_tree, y_train)
    y_pred = model.predict(X_test_tree)

    print(f"{name}: {accuracy_score(y_test, y_pred):.4f}")
    print(confusion_matrix(y_test, y_pred))
    print(classification_report(y_test, y_pred))
    print("-" * 50)

Árbol de Decisión: 0.8185
[[4339  606]
 [ 576  992]]
              precision    recall  f1-score   support

       <=50K       0.88      0.88      0.88      4945
        >50K       0.62      0.63      0.63      1568

    accuracy                           0.82      6513
   macro avg       0.75      0.76      0.75      6513
weighted avg       0.82      0.82      0.82      6513

--------------------------------------------------
Random Forest: 0.8629
[[4611  334]
 [ 559 1009]]
              precision    recall  f1-score   support

       <=50K       0.89      0.93      0.91      4945
        >50K       0.75      0.64      0.69      1568

    accuracy                           0.86      6513
   macro avg       0.82      0.79      0.80      6513
weighted avg       0.86      0.86      0.86      6513

--------------------------------------------------


## Dataset 2

In [24]:
from sklearn.linear_model import LogisticRegression
from sklearn.neighbors import KNeighborsClassifier
from sklearn.svm import SVC

models_lin = {
    'Regresión Logística': LogisticRegression(max_iter=1000, random_state=42),
    'KNN': KNeighborsClassifier(),
    'SVM': SVC(random_state=42)
}

for name, model in models_lin.items():
    model.fit(X_train_lin, y_train)
    y_pred = model.predict(X_test_lin)

    print(f"{name}: {accuracy_score(y_test, y_pred):.4f}")
    print(confusion_matrix(y_test, y_pred))
    print(classification_report(y_test, y_pred))
    print("-" * 50)

Regresión Logística: 0.8558
[[4606  339]
 [ 600  968]]
              precision    recall  f1-score   support

       <=50K       0.88      0.93      0.91      4945
        >50K       0.74      0.62      0.67      1568

    accuracy                           0.86      6513
   macro avg       0.81      0.77      0.79      6513
weighted avg       0.85      0.86      0.85      6513

--------------------------------------------------
KNN: 0.8366
[[4487  458]
 [ 606  962]]
              precision    recall  f1-score   support

       <=50K       0.88      0.91      0.89      4945
        >50K       0.68      0.61      0.64      1568

    accuracy                           0.84      6513
   macro avg       0.78      0.76      0.77      6513
weighted avg       0.83      0.84      0.83      6513

--------------------------------------------------
SVM: 0.8597
[[4656  289]
 [ 625  943]]
              precision    recall  f1-score   support

       <=50K       0.88      0.94      0.91      4945
  